In [0]:
%pip install \
numpy \
scikit-learn \
umap-learn \
optuna \
joblib \
psutil \
pandas \
tqdm
%pip install hdbscan
%pip install faiss-cpu

dbutils.library.restartPython()

In [0]:
# ============================================================

# STANDARD LIBRARIES

# ============================================================
 
import os

import glob

import time

from pathlib import Path

from multiprocessing import Manager
 
# ============================================================

# NUMERICAL / DATA

# ============================================================
 
import numpy as np

import pandas as pd

import psutil
 
# ============================================================

# VISUALIZATION

# ============================================================
 
import matplotlib.pyplot as plt
 
import plotly.io as pio

import plotly.graph_objects as go

from plotly.subplots import make_subplots
 
pio.renderers.default = "browser"
 
# ============================================================

# PROGRESS BARS

# ============================================================
 
from tqdm.auto import tqdm
 
# ============================================================

# MACHINE LEARNING

# ============================================================
 
from umap import UMAP
 
from sklearn.cluster import MiniBatchKMeans

from sklearn.metrics import silhouette_score
 
# ============================================================

# HYPERPARAMETER TUNING

# ============================================================
 
import optuna
 
# ============================================================

# OPTIONAL PARALLEL UTILITIES

# ============================================================
 
from joblib import Parallel, delayed
 



import time
import psutil
import os
import shutil
import faiss
import hdbscan

In [0]:


# CONFIG (same as your tag logic)

# ----------------------------

cache_dir = Path("/dbfs/tmp/pftsleep_cache")

encoder_name = "PFTSleep"

num_files = 1229

frequency = 125

win_length = 750

hop_length = 750

max_seq_len_sec = 8 * 3600

def cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec):

    return f"{encoder_name}__files{num_files}__freq{frequency}__win{win_length}__hop{hop_length}__max{max_seq_len_sec}"

tag = cache_tag(encoder_name, num_files, frequency, win_length, hop_length, max_seq_len_sec)

shard_dir = cache_dir / f"{tag}_shards"

shard_files = sorted(glob.glob(str(shard_dir / "Z_part_*.npy")))

assert len(shard_files) > 0, f"No shards found in {shard_dir}"

print(f"Found {len(shard_files)} shards")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 1) Compute total rows cheaply (mmap_mode avoids loading)

# ----------------------------

D = 512

total_rows = 0

first = np.load(shard_files[0], mmap_mode="r")

print("Example shard shape/dtype:", first.shape, first.dtype)

for f in shard_files:

    total_rows += np.load(f, mmap_mode="r").shape[0]

print(f"Total rows: {total_rows:,}  (expected ~5,899,200)")

print(f"Approx raw size float16: {total_rows * D * 2 / 1e9:.2f} GB")

print(f"Approx raw size float32: {total_rows * D * 4 / 1e9:.2f} GB")

# ----------------------------

# 2) Build a memmap on local NVMe (fast + avoids RAM ceilings)

# ----------------------------

local_dir = Path("/local_disk0/pftsleep_memmap")

local_dir.mkdir(parents=True, exist_ok=True)

mm_path = local_dir / f"X__{tag}__l2norm_f32.memmap"

shape_path = local_dir / f"X__{tag}__shape.txt"

# Create/overwrite memmap file

X_mm = np.memmap(mm_path, dtype=np.float32, mode="w+", shape=(total_rows, D))

# ----------------------------

# 3) Fill memmap sequentially (no vstack, no giant allocations)

# ----------------------------

t0 = time.time()

offset = 0

for f in tqdm(shard_files, desc="Writing memmap", unit="file"):

    shard = np.load(f)  # should be float16

    if shard.dtype != np.float16:

        # still fine; we cast below, but this warns you if storage isn't what you expect

        pass

    n = shard.shape[0]

    X_mm[offset:offset+n, :] = shard.astype(np.float32, copy=False)

    offset += n

X_mm.flush()

t1 = time.time()

with open(shape_path, "w") as s:

    s.write(f"{total_rows},{D}\n")

print(f"\nMemmap written: {mm_path}")

print(f"Write time: {t1 - t0:.2f} sec")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 4) In-place L2 normalize in chunks (still memmap-backed)

# ----------------------------

print("\nNormalizing memmap in chunks...")

chunk_rows = 250_000  # tune if you want (100k–500k is fine)

eps = 1e-8

t2 = time.time()

for start in tqdm(range(0, total_rows, chunk_rows), desc="L2 normalize", unit="chunk"):

    end = min(total_rows, start + chunk_rows)

    block = X_mm[start:end, :]  # view into memmap (does not load everything)

    norms = np.linalg.norm(block, axis=1, keepdims=True)

    block /= np.maximum(norms, eps)

X_mm.flush()

t3 = time.time()

print(f"Normalization time: {t3 - t2:.2f} sec")

print("✅ Memmap X is ready. Use X_mm like a normal array: X_mm[i:j]")

print(f"RAM now: {psutil.virtual_memory().available / 1e9:.2f} GB free")

# ----------------------------

# 5) Load your metadata normally (small)

# ----------------------------

night_id = np.load(cache_dir / f"night_id__{tag}.npy")

time_idx = np.load(cache_dir / f"time_idx__{tag}.npy")

zarr_file_idx = np.load(cache_dir / f"zarr_file_idx__{tag}.npy")

print("Metadata loaded:", night_id.shape, time_idx.shape, zarr_file_idx.shape)
 

In [0]:
# -------------------------
# Restructure demographics CSV to match zarr folder order
# -------------------------

# Path to zarrs folder
zarrs_dir = Path("/Workspace/Users/gpuchalski@kumc.edu/projects/PFTSleep/zarrs")

# Get all zarr files in order
zarr_files = sorted(zarrs_dir.glob("*.zarr"))
print(f"📁 Found {len(zarr_files)} zarr files")

# Extract nsrrid from each zarr filename (e.g., "shhs1-200002.zarr" -> 200002)
zarr_nsrrids = []
for zarr_file in zarr_files:
    # Extract the number after "shhs1-"
    nsrrid = int(zarr_file.stem.split('-')[-1])
    zarr_nsrrids.append(nsrrid)
    print(f"  {zarr_file.name} -> nsrrid: {nsrrid}")

# Load the original demographics CSV
demographics_df = pd.read_csv('/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel.csv')
print(f"\n📊 Original demographics shape: {demographics_df.shape}")

# Create a mapping dataframe with the desired order
order_df = pd.DataFrame({'nsrrid': zarr_nsrrids, 'order': range(len(zarr_nsrrids))})

# Merge with demographics to get the order column
demographics_ordered = demographics_df.merge(order_df, on='nsrrid', how='inner')

# Sort by the order column
demographics_ordered = demographics_ordered.sort_values('order')

# Drop the order column
demographics_ordered = demographics_ordered.drop('order', axis=1)

print(f"\n✅ Restructured demographics shape: {demographics_ordered.shape}")
print(f"📋 Matched {len(demographics_ordered)} / {len(zarr_nsrrids)} zarr files")

# Show the new order
print(f"\n🔍 First 10 nsrrids in new order:")
print(demographics_ordered['nsrrid'].head(10).tolist())

# Save the restructured CSV
output_path = '/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv'
demographics_ordered.to_csv(output_path, index=False)

print(f"\n💾 Saved restructured CSV to:")
print(f"   {output_path}")

# Verify the order matches
print(f"\n✅ Verification:")
print(f"   Zarr order: {zarr_nsrrids[:5]}")
print(f"   CSV order:  {demographics_ordered['nsrrid'].head(5).tolist()}")
print(f"   Match: {zarr_nsrrids[:5] == demographics_ordered['nsrrid'].head(5).tolist()}")

In [0]:
os.environ['OMP_NUM_THREADS'] = '16'
os.environ['NUMBA_NUM_THREADS'] = '16'
os.environ['OPENBLAS_NUM_THREADS'] = '16'

In [0]:
# ============================================================

# FAST CPU UMAP + KMeans Hyperparameter Tuning

# FAISS-accelerated neighbor graph

# Dataset: ~5.8M × 512 memmap

# ============================================================
 
import optuna

import pandas as pd

import numpy as np

import time

import psutil

import os

import faiss
faiss.omp_set_num_threads(os.cpu_count())
 
from umap import UMAP

from sklearn.cluster import MiniBatchKMeans

from sklearn.metrics import silhouette_score
 
# -------------------------------------------------

# DATASET

# -------------------------------------------------
 
print("Dataset shape:", X_mm.shape)
 
X_full = X_mm

N, D = X_full.shape
 
# -------------------------------------------------

# BUILD FAISS INDEX (RUN ONCE)

# -------------------------------------------------
 
print("\nBuilding FAISS index...")
 
t0 = time.time()
 
X_for_index = X_full
print(type(X_for_index))
print(X_for_index.shape)
print(X_for_index.dtype)

index = faiss.IndexHNSWFlat(D, 24)

index.hnsw.efConstruction = 200
print("Starting FAISS index build...")
index.add(X_for_index)
print("FAISS index build complete.")
print(f"FAISS index built in {time.time()-t0:.2f} sec")
 
# -------------------------------------------------

# PRECOMPUTE NEIGHBORS

# -------------------------------------------------
 
k_graph = 100
 
print("\nComputing nearest neighbors...")
 
t0 = time.time()
 
distances, indices = index.search(X_for_index, k_graph)
distances = np.sqrt(distances).astype(np.float32)
indices = indices.astype(np.int64)
# -------------------------------------------------
# ADD SELF-NEIGHBOR REQUIRED BY UMAP
# -------------------------------------------------
self_idx = np.arange(N).reshape(-1,1)
indices = np.concatenate([self_idx, indices[:, :-1]], axis=1)
distances = np.concatenate(
    [np.zeros((N,1), dtype=np.float32), distances[:, :-1]], axis=1
)
 
print(f"Neighbor search time: {time.time()-t0:.2f} sec")
 
# -------------------------------------------------

# SILHOUETTE SAMPLE

# -------------------------------------------------
 
rng = np.random.default_rng(42)
 
eval_size = 200_000

eval_idx = rng.choice(N, eval_size, replace=False)
 
print(f"Silhouette evaluation sample: {eval_size:,}")
 
# -------------------------------------------------

# GLOBAL TIMER

# -------------------------------------------------
 
study_start_time = time.time()
 
# -------------------------------------------------

# OPTUNA OBJECTIVE

# -------------------------------------------------
 
def objective(trial):
 
    trial_start = time.time()
 
    # -----------------------------

    # HYPERPARAMETERS

    # -----------------------------
 
    n_neighbors = trial.suggest_int("n_neighbors", 15, 80)

    min_dist = trial.suggest_float("min_dist", 0.01, 0.3, log=True)

    n_components = trial.suggest_int("n_components", 5, 20)

    k = trial.suggest_int("k", 2, 15)
    trial_start = time.time()
    # -----------------------------

    # UMAP

    # -----------------------------
 
    t0 = time.time()
    k_neighbors = n_neighbors
    reducer = UMAP(

        n_neighbors=n_neighbors,

        min_dist=min_dist,

        n_components=n_components,

        metric="euclidean",

        low_memory=True,

        transform_queue_size=4,

        random_state=None,

        verbose=True,

        n_jobs=-1,

        precomputed_knn = (
            indices[:, :k_neighbors],
            distances[:, :k_neighbors],
            None,
        )

    )
    embedding = reducer.fit_transform(X_full)
 
    umap_time = time.time() - t0
 
    print(f"UMAP finished in {umap_time/60:.2f} minutes")
 
    # -----------------------------

    # KMEANS

    # -----------------------------
 
    t0 = time.time()
 
    km = MiniBatchKMeans(

        n_clusters=k,

        batch_size=50_000,

        max_iter=100,

        n_init=5,

        random_state=42

    )
 
    labels = km.fit_predict(embedding)
 
    kmeans_time = time.time() - t0
 
    print(f"KMeans finished in {kmeans_time:.2f} sec")
 
    # -----------------------------

    # SILHOUETTE

    # -----------------------------
 
    t0 = time.time()
 
    sil = silhouette_score(

        embedding[eval_idx],

        labels[eval_idx]

    )
 
    sil_time = time.time() - t0
 
    score = float(sil)
 
    trial_total = time.time() - trial_start
 
    # -----------------------------

    # STORE TIMINGS

    # -----------------------------
 
    trial.set_user_attr("umap_time_sec", umap_time)

    trial.set_user_attr("kmeans_time_sec", kmeans_time)

    trial.set_user_attr("silhouette_time_sec", sil_time)

    trial.set_user_attr("trial_total_sec", trial_total)
    import pandas as pd
    import os
    import shutil

    local_csv = "/local_disk0/umap_trials.csv"
    dbfs_csv = "/dbfs/tmp/umap_trials.csv"
    trial_record = {

        "trial": trial.number,

        "score": score,

        "n_neighbors": n_neighbors,

        "min_dist": min_dist,

        "n_components": n_components,

        "k": k,

        "umap_time_sec": umap_time,

        "kmeans_time_sec": kmeans_time,

        "silhouette_time_sec": sil_time,

        "trial_total_sec": trial_total,

    }
 
    df = pd.DataFrame([trial_record])

    if os.path.exists(local_csv):

        df.to_csv(local_csv, mode="a", header=False, index=False)

    else:

        df.to_csv(local_csv, index=False)
 
    shutil.copy(local_csv, dbfs_csv)
    
    print(

        f"Trial {trial.number:02d} | "

        f"score={score:.4f} | "

        f"UMAP={umap_time/60:.2f}m | "

        f"KMeans={kmeans_time:.1f}s | "

        f"Sil={sil_time:.1f}s | "

        f"Total={trial_total/60:.2f}m"

    )
 
    print(f"RAM available: {psutil.virtual_memory().available/1e9:.1f} GB")
 
    return score
 
# -------------------------------------------------

# OPTUNA STUDY

# -------------------------------------------------
 
storage = "sqlite:////local_disk0/umap_optuna_cpu.db"
 
study = optuna.create_study(

    study_name="umap_kmeans_faiss_cpu_",

    storage=storage,

    load_if_exists=True,

    direction="maximize",

    sampler=optuna.samplers.TPESampler(seed=42)

)
 
print("\nStarting optimization...")
 
study.optimize(                         
    objective,

    n_trials=10,

    n_jobs=1)


 
# -------------------------------------------------

# RESULTS

# -------------------------------------------------
 
total_study_time = time.time() - study_start_time
 
print("\n========== RESULTS ==========")
 
print("Best parameters:")

print(study.best_trial.params)
 
print("\nBest silhouette:", study.best_value)
 
print(f"\nTotal study runtime: {total_study_time/3600:.2f} hours")
 
# -------------------------------------------------

# TIMING SUMMARY

# -------------------------------------------------
 
print("\nTrial timing summary:")
 
for t in study.trials:
 
    if t.value is None:

        continue
 
    print(

        f"Trial {t.number:02d} | "

        f"score={t.value:.4f} | "

        f"UMAP={t.user_attrs['umap_time_sec']/60:.2f}m | "

        f"KMeans={t.user_attrs['kmeans_time_sec']:.1f}s | "

        f"Total={t.user_attrs['trial_total_sec']/60:.2f}m"

    )
 

In [0]:
# ============================================================
# FAST CPU UMAP + HDBSCAN Hyperparameter Tuning
# FAISS-accelerated neighbor graph
# Dataset: ~5.8M × 512 memmap
# ============================================================
 
import optuna
import pandas as pd
import numpy as np
import time
import psutil
import os
import shutil
import faiss
import hdbscan
 
faiss.omp_set_num_threads(os.cpu_count())
 
from umap import UMAP
from sklearn.metrics import silhouette_score
 
# -------------------------------------------------
# DATASET
# -------------------------------------------------
 
print("Dataset shape:", X_mm.shape)
 
X_full = X_mm
N, D = X_full.shape
 
# -------------------------------------------------
# BUILD FAISS INDEX (RUN ONCE)
# -------------------------------------------------
 
print("\nBuilding FAISS index...")
 
t0 = time.time()
 
X_for_index = X_full
 
index = faiss.IndexHNSWFlat(D, 24)
index.hnsw.efConstruction = 200
 
print("Starting FAISS index build...")
index.add(X_for_index)
 
print("FAISS index build complete.")
print(f"FAISS index built in {time.time()-t0:.2f} sec")
 
# -------------------------------------------------
# PRECOMPUTE NEIGHBORS
# -------------------------------------------------
 
k_graph = 100
 
print("\nComputing nearest neighbors...")
 
t0 = time.time()
 
distances, indices = index.search(X_for_index, k_graph)
 
distances = np.sqrt(distances).astype(np.float32)
indices = indices.astype(np.int64)
 
# -------------------------------------------------
# ADD SELF NEIGHBOR (REQUIRED BY UMAP)
# -------------------------------------------------
 
self_idx = np.arange(N).reshape(-1,1)
 
indices = np.concatenate([self_idx, indices[:, :-1]], axis=1)
 
distances = np.concatenate(
    [np.zeros((N,1), dtype=np.float32), distances[:, :-1]], axis=1
)
 
print(f"Neighbor search time: {time.time()-t0:.2f} sec")
 
# -------------------------------------------------
# SILHOUETTE SAMPLE
# -------------------------------------------------
 
rng = np.random.default_rng(42)
 
eval_size = 200_000
eval_idx = rng.choice(N, eval_size, replace=False)
 
print(f"Silhouette evaluation sample: {eval_size:,}")
 
# -------------------------------------------------
# GLOBAL TIMER
# -------------------------------------------------
 
study_start_time = time.time()
 
# -------------------------------------------------
# OPTUNA OBJECTIVE
# -------------------------------------------------
 
def objective(trial):
 
    trial_start = time.time()
 
    # -----------------------------
    # HYPERPARAMETERS
    # -----------------------------
 
    n_neighbors = trial.suggest_int("n_neighbors", 5, 30)
    min_dist = trial.suggest_float("min_dist", 0.1, 0.8, log=True)
    n_components = trial.suggest_int("n_components", 10, 50)
 
    min_cluster_size = trial.suggest_int("min_cluster_size", 1000, 20000)
    min_samples = trial.suggest_int("min_samples", 10, 200)
 
    # -----------------------------
    # UMAP
    # -----------------------------
 
    t0 = time.time()
 
    reducer = UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric="euclidean",
        low_memory=True,
        transform_queue_size=4,
        random_state=None,
        verbose=True,
        n_jobs=-1,
        precomputed_knn=(
            indices[:, :n_neighbors],
            distances[:, :n_neighbors],
            None,
        ),
    )
 
    embedding = reducer.fit_transform(X_full)
 
    umap_time = time.time() - t0
 
    print(f"UMAP finished in {umap_time/60:.2f} minutes")
 
    # -----------------------------
    # HDBSCAN
    # -----------------------------
 
    t0 = time.time()
 
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        approx_min_span_tree=True,
        core_dist_n_jobs=os.cpu_count(),
        cluster_selection_epsilon=0.05,
    )
 
    labels = clusterer.fit_predict(embedding)
 
    hdbscan_time = time.time() - t0
 
    print(f"HDBSCAN finished in {hdbscan_time:.2f} sec")
 
    

    # CLUSTER-BASED METRICS (REPLACE SILHOUETTE)

    # -----------------------------
 
    labels = clusterer.labels_
 
    # Number of clusters (exclude noise)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
 
    # Noise ratio

    noise_ratio = np.mean(labels == -1)
 
    # HDBSCAN stability (main metric)

    stability = clusterer.relative_validity_
 
    # -----------------------------

    # HARD CONSTRAINTS

    # -----------------------------
 
    if n_clusters <= 1:

        return -1
 
    if n_clusters > 500:

        return -1
 
    if noise_ratio > 0.7:

        return -1
 
    if noise_ratio < 0.01:

        return -1
 
    # -----------------------------

    # FINAL SCORE

    # -----------------------------
 
    score = (

        0.5 * stability +

        0.15 * (1 - noise_ratio) +

        0.2 * sil +

        0.15 * np.log(n_clusters)

    )
 
    # -----------------------------

    # OPTIONAL: STORE METRICS

    # -----------------------------
 
    trial.set_user_attr("n_clusters", n_clusters)

    trial.set_user_attr("noise_ratio", noise_ratio)

    trial.set_user_attr("stability", stability)
 
 
    # -----------------------------
    # STORE TIMINGS
    # -----------------------------
 
    trial.set_user_attr("umap_time_sec", umap_time)
    trial.set_user_attr("hdbscan_time_sec", hdbscan_time)
    trial.set_user_attr("silhouette_time_sec", sil_time)
    trial.set_user_attr("trial_total_sec", trial_total)
 
    # -------------------------------------------------
    # SAVE TRIAL RESULTS
    # -------------------------------------------------
 
    local_csv = "/local_disk0/umap_trials_HDBSCAN_FullSet.csv"
    dbfs_csv = "/dbfs/tmp/umap_trials_HDBSCAN_FullSet.csv"
 
    trial_record = {
 
        "trial": trial.number,
        "score": score,
        "n_neighbors": n_neighbors,
        "min_dist": min_dist,
        "n_components": n_components,
        "min_cluster_size": min_cluster_size,
        "min_samples": min_samples,
        "umap_time_sec": umap_time,
        "hdbscan_time_sec": hdbscan_time,
        "silhouette_time_sec": sil_time,
        "trial_total_sec": trial_total,
        "n_clusters": n_clusters,
        "noise_ratio": noise_ratio,
        "stability": stability,
        "study_total_sec": time.time() - study_start_time,
    }
 
    df = pd.DataFrame([trial_record])
 
    if os.path.exists(local_csv):
        df.to_csv(local_csv, mode="a", header=False, index=False)
    else:
        df.to_csv(local_csv, index=False)
 
    shutil.copy(local_csv, dbfs_csv)
 
    print(
        f"Trial {trial.number:02d} | "
        f"score={score:.4f} | "
        f"UMAP={umap_time/60:.2f}m | "
        f"HDBSCAN={hdbscan_time:.1f}s | "
        f"Sil={sil_time:.1f}s | "
        f"Total={trial_total/60:.2f}m"
    )
 
    print(f"RAM available: {psutil.virtual_memory().available/1e9:.1f} GB")
 
    return score
 
 
# -------------------------------------------------
# OPTUNA STUDY
# -------------------------------------------------
 
storage = "sqlite:////local_disk0/umap_hdbscan_optuna.db"
 
study = optuna.create_study(
    study_name="umap_hdbscan_faiss_cpu",
    storage=storage,
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
 
print("\nStarting optimization...")
 
study.optimize(
    objective,
    n_trials=10,
    n_jobs=1
)
 
# -------------------------------------------------
# RESULTS
# -------------------------------------------------
 
total_study_time = time.time() - study_start_time
 
print("\n========== RESULTS ==========")
 
print("Best parameters:")
print(study.best_trial.params)
 
print("\nBest silhouette:", study.best_value)
 
print(f"\nTotal study runtime: {total_study_time/3600:.2f} hours")
 
# -------------------------------------------------
# TIMING SUMMARY
# -------------------------------------------------
 
print("\nTrial timing summary:")
 
for t in study.trials:
 
    if t.value is None:
        continue
 
    print(
        f"Trial {t.number:02d} | "
        f"score={t.value:.4f} | "
        f"UMAP={t.user_attrs['umap_time_sec']/60:.2f}m | "
        f"HDBSCAN={t.user_attrs['hdbscan_time_sec']:.1f}s | "
        f"Total={t.user_attrs['trial_total_sec']/60:.2f}m"
    )

In [0]:
# CONFIG
# ============================================================
 
KMEANS_TRIALS_CSV = "/dbfs/tmp/umap_trials_KMEANS_Fullset.csv"
HDBSCAN_TRIALS_CSV = "/dbfs/tmp/umap_trials_HDBSCAN_FullSet.csv"
 
DEMOGRAPHICS_PATH = "/Volumes/kumc_sleep/sleep_studies/shhs_data/excel_sheet/sleep_excel_ordered.csv"
OUTPUT_DIR = Path("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs_new/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
LOCAL_OUT = Path("/local_disk0/final_cluster_outputs")
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
 
VIZ_SIZE = 200_000
K_GRAPH = 100
 
# ------------------------------------------------------------
# helper: safe save to dbfs/volume by writing local then copying
# ------------------------------------------------------------
 
def save_npy_both(local_path: Path, final_path: Path, arr: np.ndarray) -> None:
    np.save(local_path, arr)
    shutil.copy(str(local_path), str(final_path))
 
def save_csv_both(local_path: Path, final_path: Path, df: pd.DataFrame) -> None:
    df.to_csv(local_path, index=True if df.index.name is not None else False)
    shutil.copy(str(local_path), str(final_path))
 

In [0]:
import threading
import time
 
def heartbeat():
    while True:
        print(f"[{time.strftime('%H:%M:%S')}] FAISS still running...")
        time.sleep(30)
 
threading.Thread(target=heartbeat, daemon=True).start()

In [0]:
# -------------------------------------------------
# DATASET
# -------------------------------------------------
 
print("Dataset shape:", X_mm.shape)
 
X_full = X_mm
N, D = X_full.shape
 
# -------------------------------------------------
# BUILD FAISS INDEX (RUN ONCE)
# -------------------------------------------------
 
print("\nBuilding FAISS index...")
 
t0 = time.time()
 
X_for_index = X_full
 
index = faiss.IndexHNSWFlat(D, 24)
index.hnsw.efConstruction = 200
 
print("Starting FAISS index build...")
index.add(X_for_index)
 
print("FAISS index build complete.")
print(f"FAISS index built in {time.time()-t0:.2f} sec")
 
# -------------------------------------------------
# PRECOMPUTE NEIGHBORS
# -------------------------------------------------
 
k_graph = 100
 
print("\nComputing nearest neighbors...")
 
t0 = time.time()
 
distances, indices = index.search(X_for_index, k_graph)
 
distances = np.sqrt(distances).astype(np.float32)
indices = indices.astype(np.int64)
 
# -------------------------------------------------
# ADD SELF NEIGHBOR (REQUIRED BY UMAP)
# -------------------------------------------------
 
self_idx = np.arange(N).reshape(-1,1)
 
indices = np.concatenate([self_idx, indices[:, :-1]], axis=1)
 
distances = np.concatenate(
    [np.zeros((N,1), dtype=np.float32), distances[:, :-1]], axis=1
)
 
print(f"Neighbor search time: {time.time()-t0:.2f} sec")

In [0]:
# ============================================================
# LOAD BEST TRIALS
# ============================================================
 
kmeans_trials = pd.read_csv(KMEANS_TRIALS_CSV)
hdbscan_trials = pd.read_csv(HDBSCAN_TRIALS_CSV)
 
best_kmeans = kmeans_trials.loc[kmeans_trials["score"].idxmax()].copy()
best_hdbscan = hdbscan_trials.loc[hdbscan_trials["score"].idxmax()].copy()
 
print("\nBest KMeans trial:")
print(best_kmeans)
 
print("\nBest HDBSCAN trial:")
print(best_hdbscan)
 
# ============================================================
# FIT FINAL FULL UMAP EMBEDDINGS
# ============================================================
 
def fit_full_umap(best_row: pd.Series, name: str) -> np.ndarray:
    n_neighbors = int(best_row["n_neighbors"])
    min_dist = float(best_row["min_dist"])
    n_components = int(best_row["n_components"])
 
    print(f"\nRunning final full UMAP for {name}...")
    print(
        f"n_neighbors={n_neighbors}, min_dist={min_dist}, "
        f"n_components={n_components}"
    )
 
    t0 = time.time()
 
    reducer = UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric="euclidean",
        low_memory=True,
        transform_queue_size=4,
        random_state=None,
        verbose=False,
        n_jobs=-1,
        precomputed_knn=(
            indices[:, :n_neighbors],
            distances[:, :n_neighbors],
            None,
        ),
    )
 
    embedding = reducer.fit_transform(X_full)
 
    print(f"{name} full UMAP finished in {(time.time()-t0)/60:.2f} minutes")
    print(f"{name} embedding shape: {embedding.shape}")
    return embedding.astype(np.float32, copy=False)
 
embedding_kmeans = fit_full_umap(best_kmeans, "KMeans")
embedding_hdbscan = fit_full_umap(best_hdbscan, "HDBSCAN")
 
# save embeddings
kmeans_embedding_final = OUTPUT_DIR / f"umap_embedding_kmeans_best__{tag}.npy"
hdbscan_embedding_final = OUTPUT_DIR / f"umap_embedding_hdbscan_best__{tag}.npy"
 
save_npy_both(
    LOCAL_OUT / f"umap_embedding_kmeans_best__{tag}.npy",
    kmeans_embedding_final,
    embedding_kmeans
)
save_npy_both(
    LOCAL_OUT / f"umap_embedding_hdbscan_best__{tag}.npy",
    hdbscan_embedding_final,
    embedding_hdbscan
)
 
print("Saved embeddings.")
 

In [0]:
# ============================================================
# FINAL KMEANS ON FULL UMAP EMBEDDING
# ============================================================
 
best_k = int(best_kmeans["k"])
 
print(f"\nFitting final MiniBatchKMeans with k={best_k}...")
t0 = time.time()
 
final_km = MiniBatchKMeans(
    n_clusters=best_k,
    batch_size=50_000,
    max_iter=200,
    n_init=5,
    random_state=42
)
 
cluster_id_kmeans = final_km.fit_predict(embedding_kmeans)
 
print(f"Final KMeans complete in {time.time() - t0:.1f}s")
 
kmeans_labels_path = OUTPUT_DIR / f"cluster_id_kmeans__{tag}.npy"
save_npy_both(
    LOCAL_OUT / f"cluster_id_kmeans__{tag}.npy",
    kmeans_labels_path,
    cluster_id_kmeans
)
print("KMeans cluster IDs saved.")
 
unique, counts = np.unique(cluster_id_kmeans, return_counts=True)
print("\nKMeans cluster counts:")
for u, c in zip(unique, counts):
    print(f"Cluster {u}: {c:,} windows")
 

In [0]:
# ============================================================
# FINAL HDBSCAN ON FULL UMAP EMBEDDING
# ============================================================
 
best_min_cluster_size = int(best_hdbscan["min_cluster_size"])
best_min_samples = int(best_hdbscan["min_samples"])
 
print(
    f"\nFitting final HDBSCAN with "
    f"min_cluster_size={best_min_cluster_size}, "
    f"min_samples={best_min_samples}..."
)
t0 = time.time()
 
final_hdb = hdbscan.HDBSCAN(
    min_cluster_size=best_min_cluster_size,
    min_samples=best_min_samples,
    metric="euclidean",
    cluster_selection_method="eom",
    approx_min_span_tree=True,
    core_dist_n_jobs=os.cpu_count()
)
 
cluster_id_hdbscan = final_hdb.fit_predict(embedding_hdbscan)
 
print(f"Final HDBSCAN complete in {time.time() - t0:.1f}s")
 
hdbscan_labels_path = OUTPUT_DIR / f"cluster_id_hdbscan__{tag}.npy"
save_npy_both(
    LOCAL_OUT / f"cluster_id_hdbscan__{tag}.npy",
    hdbscan_labels_path,
    cluster_id_hdbscan
)
print("HDBSCAN cluster IDs saved.")
 
unique, counts = np.unique(cluster_id_hdbscan, return_counts=True)
print("\nHDBSCAN cluster counts:")
for u, c in zip(unique, counts):
    print(f"Cluster {u}: {c:,} windows")


In [0]:
# ============================================================
# DEMOGRAPHICS + SUBJECT-LEVEL FRACTIONS
# ============================================================
 
demographics_df = pd.read_csv(DEMOGRAPHICS_PATH)
demographics_df["nsrrid"] = demographics_df["nsrrid"].astype(int)
 
print("\nLoaded demographics:", demographics_df.shape)
 
def build_subject_outputs(cluster_id: np.ndarray, method_name: str):
    df = pd.DataFrame({
        "cluster": cluster_id,
        "nsrrid": zarr_file_idx
    })
 
    cluster_fraction = (
        df.groupby("nsrrid")["cluster"]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
    )
 
    cluster_fraction.index = zarr_nsrrids
    cluster_fraction.index.name = "nsrrid"
 
    cluster_demo = (
        cluster_fraction
        .reset_index()
        .merge(demographics_df, on="nsrrid", how="left")
    )
 
    cluster_demo["dominant_cluster"] = cluster_fraction.idxmax(axis=1).values
 
    def summarize_cluster(subdf):
        return {
            "N": len(subdf),
            "Age_mean": subdf["age_s1"].mean(),
            "Age_min": subdf["age_s1"].min(),
            "Age_max": subdf["age_s1"].max(),
            "BMI_mean": subdf["bmi_s1"].mean(),
            "BMI_min": subdf["bmi_s1"].min(),
            "BMI_max": subdf["bmi_s1"].max(),
            "AHI_mean": subdf["ahi_a0h4_s1"].mean(),
            "AHI_min": subdf["ahi_a0h4_s1"].min(),
            "AHI_max": subdf["ahi_a0h4_s1"].max(),
            "ESS_mean": subdf["ess_s1"].mean(),
            "MinSat_mean": subdf["MinSat"].mean(),
            "PctLT90_mean": subdf["pctlt90"].mean(),
            "Diabetes_%": (subdf["Diabetes"] == "Yes").mean() * 100,
            "Prev_CVD_%": (subdf["prev_cvd_all_01"] == "Yes").mean() * 100,
            "Male_%": (subdf["gender"] == "Male").mean() * 100,
            "Smoker_%": (subdf["smokecat_s1"] == "Current").mean() * 100,
        }
 
    summary_table = (
        cluster_demo
        .groupby("dominant_cluster")
        .apply(summarize_cluster)
        .apply(pd.Series)
    )
 
    # save outputs
    frac_path = OUTPUT_DIR / f"cluster_fraction_{method_name}__{tag}.csv"
    demo_path = OUTPUT_DIR / f"cluster_demo_{method_name}__{tag}.csv"
    summary_path = OUTPUT_DIR / f"cluster_summary_{method_name}__{tag}.csv"
 
    save_csv_both(
        LOCAL_OUT / f"cluster_fraction_{method_name}__{tag}.csv",
        frac_path,
        cluster_fraction
    )
 
    save_csv_both(
        LOCAL_OUT / f"cluster_demo_{method_name}__{tag}.csv",
        demo_path,
        cluster_demo.set_index("nsrrid")
    )
 
    save_csv_both(
        LOCAL_OUT / f"cluster_summary_{method_name}__{tag}.csv",
        summary_path,
        summary_table
    )
 
    print(f"\nSaved {method_name} outputs to:")
    print(frac_path)
    print(demo_path)
    print(summary_path)
 
    print(f"Merged subjects ({method_name}):", len(cluster_demo))
    print(f"Unique nsrrid ({method_name}):", cluster_demo['nsrrid'].nunique())
 
    return cluster_fraction, cluster_demo, summary_table
 
cluster_fraction_kmeans, cluster_demo_kmeans, summary_kmeans = build_subject_outputs(
    cluster_id_kmeans, "kmeans"
)
cluster_fraction_hdbscan, cluster_demo_hdbscan, summary_hdbscan = build_subject_outputs(
    cluster_id_hdbscan, "hdbscan"
)
 
print("\nKMeans summary table:")
display(summary_kmeans)
 
print("\nHDBSCAN summary table:")
display(summary_hdbscan)
 

In [0]:
embedding_kmeans = np.load("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs_new/umap_embedding_kmeans_best__PFTSleep__files1229__freq125__win750__hop750__max28800.npy")
cluster_id_kmeans = np.load("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs_new/cluster_id_kmeans__PFTSleep__files1229__freq125__win750__hop750__max28800.npy")
cluster_id_hdbscan = np.load("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs_new/cluster_id_hdbscan__PFTSleep__files1229__freq125__win750__hop750__max28800.npy")
cluster_demo_kmeans = pd.read_csv("/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs_new/cluster_demo_kmeans__PFTSleep__files1229__freq125__win750__hop750__max28800.csv")

In [0]:
# ============================================================

# FINAL 3D VISUALIZATION (UMAP-on-UMAP)

# - Uses FINAL clustering labels (no reclustering)

# - Fast: operates on 35D embedding

# ============================================================
 
from umap import UMAP

from mpl_toolkits.mplot3d import Axes3D  # noqa
 
print("\nBuilding 3D visualization embedding (UMAP-on-UMAP)...")
 
t0 = time.time()
 
reducer_viz = UMAP(

    n_neighbors=30,

    min_dist=0.05,

    n_components=3,

    metric="euclidean",

    random_state=42,

    n_jobs=-1,

    low_memory=True,

)
 
# Use SAME embedding used for clustering

embedding_3d = reducer_viz.fit_transform(embedding_kmeans)
 
print(f"3D embedding done in {time.time() - t0:.2f} sec")

print("Embedding shape:", embedding_3d.shape)
 
# ------------------------------------------------------------

# SUBSAMPLE FOR VISUALIZATION

# ------------------------------------------------------------
 
rng = np.random.default_rng(42)
 
viz_size = min(200_000, len(cluster_id_kmeans))

viz_idx = rng.choice(len(cluster_id_kmeans), viz_size, replace=False)
 
emb_viz = embedding_3d[viz_idx]
 
labels_viz_kmeans = cluster_id_kmeans[viz_idx]

labels_viz_hdbscan = cluster_id_hdbscan[viz_idx]
 
# Highlight HDBSCAN noise

labels_viz_hdbscan_plot = labels_viz_hdbscan.copy()

labels_viz_hdbscan_plot[labels_viz_hdbscan_plot == -1] = -99
 
print(f"Visualization sample size: {viz_size:,}")
 
# ------------------------------------------------------------

# MAP AHI

# ------------------------------------------------------------
 
window_file_idx = zarr_file_idx[viz_idx]

window_nsrrids = np.array(zarr_nsrrids)[window_file_idx]
 
ahi_lookup = cluster_demo_kmeans.set_index("nsrrid")["ahi_a0h4_s1"].to_dict()
 
ahi_viz = np.array([

    ahi_lookup.get(nsrrid, np.nan)

    for nsrrid in window_nsrrids

])
 
print("AHI mapping complete.")
 
# ------------------------------------------------------------

# PLOTTING FUNCTION

# ------------------------------------------------------------
 
def plot_3d(emb, labels, title, cmap="tab20"):
 
    fig = plt.figure(figsize=(10, 8))

    ax = fig.add_subplot(111, projection='3d')
 
    sc = ax.scatter(

        emb[:, 0],

        emb[:, 1],

        emb[:, 2],

        c=labels,

        s=3,

        alpha=0.6,

        cmap=cmap

    )
 
    ax.set_title(title)

    ax.set_xlabel("UMAP-1")

    ax.set_ylabel("UMAP-2")

    ax.set_zlabel("UMAP-3")
 
    plt.colorbar(sc)

    plt.tight_layout()
    plt.savefig(f"/Volumes/kumc_sleep/sleep_studies/shhs_data/clustering_outputs_new/{title}.png")
    plt.show()
 
# ------------------------------------------------------------

# PLOTS

# ------------------------------------------------------------
 
print("\nPlotting KMeans (3D)...")

plot_3d(

    emb_viz,

    labels_viz_kmeans,

    f"KMeans Clusters (k={len(np.unique(cluster_id_kmeans))})"

)
 
print("\nPlotting HDBSCAN (3D)...")

plot_3d(

    emb_viz,

    labels_viz_hdbscan_plot,

    "HDBSCAN Clusters (noise shown separately)",

    cmap="tab20"

)
 
print("\nPlotting AHI (3D)...")

plot_3d(

    emb_viz,

    ahi_viz,

    "UMAP Colored by AHI",

    cmap="viridis"

)
 
print("\nDone.")
 

In [0]:
# ============================================================
# VISUALIZATION HELPERS
# Use sampled rows from FINAL UMAP embedding (CPU-friendly)
# ============================================================
 
rng = np.random.default_rng(42)
viz_size = min(VIZ_SIZE, N)
viz_idx = rng.choice(N, viz_size, replace=False)
 
print(f"\nVisualization sample size: {viz_size:,}")
 
def build_viz_payload(
    embedding_full: np.ndarray,
    cluster_id: np.ndarray,
    cluster_demo: pd.DataFrame,
    title_prefix: str
):
    embedding_viz = embedding_full[viz_idx]
    labels_viz = cluster_id[viz_idx]
 
    # map window -> subject -> AHI
    window_file_idx = zarr_file_idx[viz_idx]
    window_nsrrids = np.array(zarr_nsrrids)[window_file_idx]
 
    ahi_lookup = cluster_demo.set_index("nsrrid")["ahi_a0h4_s1"].to_dict()
    ahi_viz = np.array([ahi_lookup.get(nsrrid, np.nan) for nsrrid in window_nsrrids])
 
    print(f"{title_prefix}: AHI mapping complete.")
    return embedding_viz, labels_viz, ahi_viz
 
emb_viz_kmeans, labels_viz_kmeans, ahi_viz_kmeans = build_viz_payload(
    embedding_kmeans, cluster_id_kmeans, cluster_demo_kmeans, "KMeans"
)
emb_viz_hdbscan, labels_viz_hdbscan, ahi_viz_hdbscan = build_viz_payload(
    embedding_hdbscan, cluster_id_hdbscan, cluster_demo_hdbscan, "HDBSCAN"
)
 
# ============================================================
# PLOTS
# ============================================================
 
def plot_embedding_views(embedding, labels, ahi_viz, title_prefix):
    # 2D scatter colored by cluster
    plt.figure(figsize=(10, 8))
    sc = plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=labels,
        s=3,
        alpha=0.7,
        cmap="tab20"
    )
    plt.title(f"{title_prefix} UMAP Colored by Cluster")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.colorbar(sc, label="Cluster")
    plt.tight_layout()
    plt.show()
 
    # density hexbin with mean cluster id
    plt.figure(figsize=(10, 8))
    hb = plt.hexbin(
        embedding[:, 0],
        embedding[:, 1],
        gridsize=150,
        C=labels.astype(float),
        reduce_C_function=np.mean,
        mincnt=1
    )
    plt.colorbar(hb, label="Mean Cluster ID")
    plt.title(f"{title_prefix} UMAP Density View")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.tight_layout()
    plt.show()
 
    # colored by AHI
    plt.figure(figsize=(10, 8))
    sc2 = plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=ahi_viz,
        s=3,
        alpha=0.7
    )
    plt.colorbar(sc2, label="AHI")
    plt.title(f"{title_prefix} UMAP Colored by AHI")
    plt.xlabel("UMAP-1")
    plt.ylabel("UMAP-2")
    plt.tight_layout()
    plt.show()
 
print("\nPlotting KMeans views...")
plot_embedding_views(emb_viz_kmeans, labels_viz_kmeans, ahi_viz_kmeans, "KMeans")
 
print("\nPlotting HDBSCAN views...")
plot_embedding_views(emb_viz_hdbscan, labels_viz_hdbscan, ahi_viz_hdbscan, "HDBSCAN")
 
print("\nDone.")